# A0049000 FIRM 확정 기준 4주 예측 v8

내부데이터만 / 외부데이터만 / 내부+외부 통합 3가지 모델 비교 코드입니다. 코랩에서 이 셀 하나만 실행하면 됩니다.

In [ ]:
# ============================================================
# A0049000 FIRM 확정 기준 4주 예측 v8 - 3가지 데이터셋 비교
# 목적:
# 1) 내부데이터만 사용한 모델
# 2) 외부데이터만 사용한 모델
# 3) 내부+외부 통합 모델
# 각 그룹에서 calibration 기준으로 최적 후보를 선택하고 holdout 성능을 비교
# - PASS 컬럼 없음
# - RMSE / WAPE / MAE / Sum Gap Rate 수치만 출력
# - 외부변수 lag 1~16 유지
# ============================================================

import warnings
warnings.filterwarnings('ignore')

import os
import re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import RidgeCV, ElasticNet
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error

try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    IN_COLAB = False

# ------------------------------------------------------------
# 0. 경로 / 설정
# ------------------------------------------------------------
# 코랩에서는 /content, 로컬/ChatGPT 실행환경에서는 현재 폴더 또는 /mnt/data를 자동 탐색
TARGET_NAME = 'A0049000_최신데이터.xlsx'
EXTERNAL_NAME = '외부변수 통합 데이터들.xlsx'

if Path('/content/' + TARGET_NAME).exists():
    DEMAND_PATH = Path('/content/' + TARGET_NAME)
elif Path(TARGET_NAME).exists():
    DEMAND_PATH = Path(TARGET_NAME)
else:
    DEMAND_PATH = Path('/content/' + TARGET_NAME)

if Path('/content/' + EXTERNAL_NAME).exists():
    EXTERNAL_PATH = Path('/content/' + EXTERNAL_NAME)
elif Path(EXTERNAL_NAME).exists():
    EXTERNAL_PATH = Path(EXTERNAL_NAME)
else:
    EXTERNAL_PATH = Path('/content/' + EXTERNAL_NAME)

TARGET_PN = 'A0049000'
TARGET_COL = 'Quantity'
FIRM_VALUE = 'FIRM'
FORECAST_VALUE = 'FORECAST'
DATE_COL_EXTERNAL = 'Date'

FORECAST_WEEKS = 4
MAX_EXTERNAL_LAG = 16
HOLDOUT_CONFIRMED_WEEKS = 8
INNER_CAL_CONFIRMED_WEEKS = 4
OUTPUT_PREFIX = f'{TARGET_PN}_firm_confirmed_v8_3datasets'

# pack 단위: 기존 데이터가 0.896 단위로 움직이는 형태라 예측값도 같은 단위로 정리
PACK_UNIT = 0.896

MONTH_MAP = {
    'JANUARY':1,'JAN':1,'1월':1,'01월':1,
    'FEBRUARY':2,'FEB':2,'2월':2,'02월':2,
    'MARCH':3,'MAR':3,'3월':3,'03월':3,
    'APRIL':4,'APR':4,'4월':4,'04월':4,
    'MAY':5,'5월':5,'05월':5,
    'JUNE':6,'JUN':6,'6월':6,'06월':6,
    'JULY':7,'JUL':7,'7월':7,'07월':7,
    'AUGUST':8,'AUG':8,'8월':8,'08월':8,
    'SEPTEMBER':9,'SEP':9,'SEPT':9,'9월':9,'09월':9,
    'OCTOBER':10,'OCT':10,'10월':10,
    'NOVEMBER':11,'NOV':11,'11월':11,
    'DECEMBER':12,'DEC':12,'12월':12,
}

# ------------------------------------------------------------
# 1. 유틸
# ------------------------------------------------------------
def ensure_files_exist():
    missing = [p for p in [DEMAND_PATH, EXTERNAL_PATH] if not p.exists()]
    if missing and IN_COLAB:
        print('아래 파일 2개를 업로드하세요:')
        print('1) A0049000_최신데이터.xlsx')
        print('2) 외부변수 통합 데이터들.xlsx')
        files.upload()
    missing = [p for p in [DEMAND_PATH, EXTERNAL_PATH] if not p.exists()]
    if missing:
        raise FileNotFoundError('필수 파일 없음: ' + ', '.join(str(p) for p in missing))


def clean_col(c):
    return str(c).replace('\n',' ').replace('\r',' ').strip()


def norm(s):
    return re.sub(r'[^a-z0-9가-힣]+', '', str(s).lower())


def find_col(cols, names, contains=None):
    nmap = {norm(c): c for c in cols}
    for name in names:
        if norm(name) in nmap:
            return nmap[norm(name)]
    if contains:
        for c in cols:
            if any(norm(x) in norm(c) for x in contains):
                return c
    return None


def parse_num(s):
    if pd.isna(s):
        return np.nan
    if isinstance(s, (int, float, np.integer, np.floating)):
        return float(s)
    t = str(s).strip().replace(',', '')
    if t.count('.') >= 2:
        t = t.replace('.', '')
    return pd.to_numeric(t, errors='coerce')


def make_date(df):
    m_raw = df['Month'].astype(str).str.strip()
    m = m_raw.str.upper().map(MONTH_MAP)
    m = m.fillna(pd.to_numeric(m_raw, errors='coerce'))
    return pd.to_datetime({
        'year': pd.to_numeric(df['Year'], errors='coerce'),
        'month': m,
        'day': pd.to_numeric(df['Day'], errors='coerce')
    }, errors='coerce')


def week_start_monday(s):
    s = pd.to_datetime(s)
    return s - pd.to_timedelta(s.dt.weekday, unit='D')


def rmse(y, p):
    y = np.asarray(y, dtype=float); p = np.asarray(p, dtype=float)
    return float(np.sqrt(np.mean((y-p)**2)))


def wape(y, p):
    y = np.asarray(y, dtype=float); p = np.asarray(p, dtype=float)
    den = np.sum(np.abs(y))
    return np.nan if den == 0 else float(np.sum(np.abs(y-p)) / den)


def mase(y, p, train_y):
    train_y = np.asarray(train_y, dtype=float)
    if len(train_y) < 2:
        return np.nan
    naive = np.mean(np.abs(train_y[1:] - train_y[:-1]))
    return np.nan if naive == 0 else float(np.mean(np.abs(np.asarray(y)-np.asarray(p))) / naive)


def sum_gap_rate(y, p):
    ysum = np.sum(y)
    return np.nan if ysum == 0 else float((np.sum(p)-ysum)/ysum)


def pack_round(x, unit=PACK_UNIT):
    arr = np.asarray(x, dtype=float)
    arr = np.maximum(arr, 0)
    return np.round(arr / unit) * unit


def eval_row(dataset_type, model_name, y, p, train_y, scale=1.0, offset=0.0):
    p = np.maximum(np.asarray(p, dtype=float), 0)
    return {
        'Dataset_Type': dataset_type,
        'Model': model_name,
        'Row_N': len(y),
        'RMSE': rmse(y, p),
        'WAPE': wape(y, p),
        'MAE': float(mean_absolute_error(y, p)),
        'MASE': mase(y, p, train_y),
        'Sum Gap Rate': sum_gap_rate(y, p),
        'Actual Sum': float(np.sum(y)),
        'Pred Sum': float(np.sum(p)),
        'Scale': scale,
        'Offset': offset,
    }


def score_for_select(y, p, train_y):
    # RMSE와 WAPE를 동시에 낮추되, 둘 중 하나가 튀면 불리하게 함
    r = rmse(y, p)
    w = wape(y, p)
    g = abs(sum_gap_rate(y, p))
    return 0.60*r + 1.20*w + 0.20*g


def best_scale_offset(y, raw_pred, allow_offset=True):
    # calibration에서만 scale/offset 탐색. holdout/test는 건드리지 않음.
    y = np.asarray(y, dtype=float)
    raw_pred = np.asarray(raw_pred, dtype=float)
    best = (np.inf, 1.0, 0.0)
    scales = np.round(np.arange(0.60, 1.51, 0.01), 2)
    offsets = np.arange(-1.5, 1.51, 0.25) if allow_offset else [0.0]
    for s in scales:
        for b in offsets:
            p = pack_round(np.maximum(raw_pred * s + b, 0))
            sc = score_for_select(y, p, y)
            if sc < best[0]:
                best = (sc, float(s), float(b))
    return best[1], best[2]

# ------------------------------------------------------------
# 2. 데이터 로드
# ------------------------------------------------------------
def load_demand_weekly(path):
    raw = pd.read_excel(path)
    raw.columns = [clean_col(c) for c in raw.columns]
    if 'ZF_PN' in raw.columns:
        raw = raw[raw['ZF_PN'].astype(str).str.strip().eq(TARGET_PN)].copy()

    firm_col = find_col(raw.columns, ['Firm/Forecast', 'Firm Forecast'], contains=['firm','forecast'])
    if firm_col is None:
        raise ValueError('Firm/Forecast 컬럼을 찾지 못했습니다.')

    raw['date'] = make_date(raw)
    raw[TARGET_COL] = pd.to_numeric(raw[TARGET_COL], errors='coerce')
    raw['CUM_QTY_NUM'] = raw['CUM_QTY'].apply(parse_num) if 'CUM_QTY' in raw.columns else np.nan
    raw = raw.dropna(subset=['date', TARGET_COL]).copy()
    raw['week'] = week_start_monday(raw['date'])
    raw[firm_col] = raw[firm_col].astype(str).str.upper().str.strip()

    aggs = []
    for status in [FIRM_VALUE, FORECAST_VALUE]:
        part = raw[raw[firm_col].eq(status)].copy()
        g = part.groupby('week').agg(
            qty=(TARGET_COL, 'sum'),
            cnt=(TARGET_COL, 'size'),
            mean=(TARGET_COL, 'mean'),
            max=(TARGET_COL, 'max'),
            std=(TARGET_COL, 'std'),
            order_n=('Order', 'nunique') if 'Order' in part.columns else (TARGET_COL, 'size'),
            cum_max=('CUM_QTY_NUM', 'max'),
        )
        g = g.add_prefix(status.lower() + '_')
        aggs.append(g)

    weekly = pd.concat(aggs, axis=1).reset_index()
    full = pd.DataFrame({'week': pd.date_range(weekly['week'].min(), weekly['week'].max(), freq='W-MON')})
    weekly = full.merge(weekly, on='week', how='left').fillna(0)
    weekly = weekly.rename(columns={'firm_qty':'firm_quantity', 'forecast_qty':'forecast_quantity'})

    confirmed_weeks = weekly.loc[weekly['firm_quantity'] > 0, 'week'].copy()
    return weekly, raw, firm_col, confirmed_weeks


def load_external_weekly(path):
    ext = pd.read_excel(path)
    ext.columns = [clean_col(c) for c in ext.columns]
    date_col = find_col(ext.columns, [DATE_COL_EXTERNAL, 'Date', '날짜'], contains=['date','날짜'])
    if date_col is None:
        raise ValueError('외부변수 Date 컬럼을 찾지 못했습니다.')
    ext[date_col] = pd.to_datetime(ext[date_col], errors='coerce')
    ext = ext.dropna(subset=[date_col]).copy()
    ext['week'] = week_start_monday(ext[date_col])
    num_cols = [c for c in ext.columns if c not in [date_col, 'week']]
    for c in num_cols:
        ext[c] = pd.to_numeric(ext[c], errors='coerce')
    num_cols = [c for c in num_cols if pd.api.types.is_numeric_dtype(ext[c])]
    weekly_ext = ext.groupby('week', as_index=False)[num_cols].mean().sort_values('week')
    return weekly_ext, num_cols


def make_base(weekly, weekly_ext):
    last_confirmed = weekly.loc[weekly['firm_quantity'] > 0, 'week'].max()
    future_weeks = list(pd.date_range(last_confirmed + pd.Timedelta(days=7), periods=FORECAST_WEEKS, freq='W-MON'))
    max_week = max(weekly['week'].max(), future_weeks[-1])
    base = pd.DataFrame({'week': pd.date_range(weekly['week'].min(), max_week, freq='W-MON')})
    df = base.merge(weekly, on='week', how='left')
    df = df.merge(weekly_ext, on='week', how='left')
    fill_cols = [c for c in df.columns if c != 'week']
    df[fill_cols] = df[fill_cols].ffill().bfill().fillna(0)
    return df, future_weeks

# ------------------------------------------------------------
# 3. Feature Engineering
# ------------------------------------------------------------
def engineer_features(df, ext_cols):
    df = df.copy().sort_values('week').reset_index(drop=True)
    df['weekofyear'] = df['week'].dt.isocalendar().week.astype(int)
    df['month_num'] = df['week'].dt.month

    # 내부 데이터: firm / forecast 기반 feature
    for lag in range(1, 17):
        df[f'firm_lag{lag}'] = df['firm_quantity'].shift(lag)
        df[f'forecast_lag{lag}'] = df['forecast_quantity'].shift(lag)
        df[f'forecast_cnt_lag{lag}'] = df['forecast_cnt'].shift(lag) if 'forecast_cnt' in df.columns else 0
        df[f'forecast_mean_lag{lag}'] = df['forecast_mean'].shift(lag) if 'forecast_mean' in df.columns else 0
        df[f'forecast_max_lag{lag}'] = df['forecast_max'].shift(lag) if 'forecast_max' in df.columns else 0

    for win in [2,3,4,6,8,12,16]:
        df[f'firm_ma{win}'] = df['firm_quantity'].shift(1).rolling(win).mean()
        df[f'firm_med{win}'] = df['firm_quantity'].shift(1).rolling(win).median()
        df[f'forecast_ma{win}'] = df['forecast_quantity'].shift(1).rolling(win).mean()
        df[f'forecast_med{win}'] = df['forecast_quantity'].shift(1).rolling(win).median()
        ratio = df['firm_quantity'].shift(1) / df['forecast_quantity'].shift(1).replace(0, np.nan)
        df[f'ratio_med{win}'] = ratio.rolling(win).median()
        df[f'ratio_mean{win}'] = ratio.rolling(win).mean()
        df[f'ratio_pred_med{win}'] = df['forecast_quantity'] * df[f'ratio_med{win}']
        df[f'ratio_pred_mean{win}'] = df['forecast_quantity'] * df[f'ratio_mean{win}']

    df['firm_trend_1_2'] = df['firm_lag1'] - df['firm_lag2']
    df['firm_trend_4'] = df['firm_lag1'] - df['firm_lag4']
    df['forecast_chg1'] = df['forecast_quantity'] - df['forecast_lag1']
    df['forecast_pct_chg1'] = df['forecast_chg1'] / df['forecast_lag1'].replace(0, np.nan)
    df['forecast_current'] = df['forecast_quantity']
    df['forecast_to_ma4'] = df['forecast_quantity'] / df['forecast_ma4'].replace(0, np.nan)

    # 외부 데이터: 외부변수 lag 1~16 + MA4 + 변화율
    for c in ext_cols:
        for lag in range(1, MAX_EXTERNAL_LAG + 1):
            df[f'EXT_L{lag}__{c}'] = df[c].shift(lag)
        df[f'EXT_MA4__{c}'] = df[c].shift(1).rolling(4).mean()
        df[f'EXT_CHG1__{c}'] = df[c].shift(1).pct_change(1)

    return df.replace([np.inf, -np.inf], np.nan)


def simple_internal_candidates(df):
    # 내부 데이터만으로 만들 수 있는 휴리스틱 후보
    out = pd.DataFrame({'week': df['week']})
    out['fixed_ma'] = 0.50*df['firm_lag1'] + 0.30*df['firm_ma4'] + 0.20*df['firm_ma8']
    out['recent_ma'] = 0.70*df['firm_lag1'] + 0.30*df['firm_ma4']
    out['conservative'] = 0.60*df['firm_lag1'] + 0.20*df['firm_ma4'] + 0.20*df['firm_med12']
    for w in [2,3,4,6,8,12,16]:
        out[f'ratio_med{w}'] = df[f'ratio_pred_med{w}']
        out[f'ratio_mean{w}'] = df[f'ratio_pred_mean{w}']
    return out.drop(columns=['week']).clip(lower=0)


def feature_groups(feature_cols):
    calendar = [c for c in feature_cols if c in ['weekofyear', 'month_num']]
    external = [c for c in feature_cols if c.startswith('EXT_')]
    # 원본 외부변수명은 feature_cols에 남아있을 수 있으므로 제외하지 않고 EXT 기반만 외부 feature로 사용
    internal = [c for c in feature_cols if c not in external and c not in calendar]
    return internal, external, calendar


def get_feature_set(all_feature_cols, dataset_type):
    internal, external, calendar = feature_groups(all_feature_cols)
    if dataset_type == 'internal_only':
        return internal + calendar
    if dataset_type == 'external_only':
        return external + calendar
    if dataset_type == 'combined':
        return internal + external + calendar
    raise ValueError(f'알 수 없는 dataset_type: {dataset_type}')

# ------------------------------------------------------------
# 4. 모델 학습 / 후보 평가
# ------------------------------------------------------------
def fit_ml_models(train_df, feature_cols):
    X = train_df[feature_cols]
    y = train_df['firm_quantity']
    models = {
        'ml_ridge': Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', RobustScaler()), ('m', RidgeCV(alphas=[0.01,0.1,1,3,10,30,100,300]))]),
        'ml_elastic': Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler()), ('m', ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=10000, random_state=42))]),
        'ml_rf': Pipeline([('imp', SimpleImputer(strategy='median')), ('m', RandomForestRegressor(n_estimators=120, max_depth=4, min_samples_leaf=2, random_state=42, n_jobs=-1))]),
        'ml_extratrees': Pipeline([('imp', SimpleImputer(strategy='median')), ('m', ExtraTreesRegressor(n_estimators=120, max_depth=4, min_samples_leaf=2, random_state=43, n_jobs=-1))]),
        'ml_gbr': Pipeline([('imp', SimpleImputer(strategy='median')), ('m', GradientBoostingRegressor(n_estimators=80, max_depth=1, learning_rate=0.05, random_state=44))]),
        'ml_knn5': Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler()), ('m', KNeighborsRegressor(n_neighbors=5, weights='distance'))]),
    }
    fitted = {}
    for name, model in models.items():
        try:
            model.fit(X, y)
            fitted[name] = model
        except Exception as e:
            print('[모델 스킵]', name, e)
    return fitted


def predict_candidates(dataset_type, fit_df, pred_df, feature_cols):
    cand = pd.DataFrame({'week': pred_df['week'].values})

    # 내부/통합 모델에서는 내부 휴리스틱 후보도 같이 비교
    if dataset_type in ['internal_only', 'combined']:
        simple = simple_internal_candidates(pred_df).reset_index(drop=True)
        cand = pd.concat([cand, simple], axis=1)

    # 모든 dataset_type에서 ML 후보 사용
    if len(feature_cols) > 0:
        models = fit_ml_models(fit_df, feature_cols)
        Xp = pred_df[feature_cols]
        for name, model in models.items():
            cand[name] = np.maximum(model.predict(Xp), 0)
    else:
        models = {}

    return cand, models


def choose_best_by_cal(dataset_type, fit, cal, feature_cols):
    cal_cand, models = predict_candidates(dataset_type, fit, cal, feature_cols)
    rows = []
    params = {}

    for col in [c for c in cal_cand.columns if c != 'week']:
        raw = np.asarray(cal_cand[col], dtype=float)
        s, b = best_scale_offset(cal['firm_quantity'], raw, allow_offset=True)
        p = pack_round(np.maximum(raw*s + b, 0))
        row = eval_row(dataset_type, col, cal['firm_quantity'], p, fit['firm_quantity'], s, b)
        row['SelectScore'] = score_for_select(cal['firm_quantity'], p, fit['firm_quantity'])
        rows.append(row)
        params[col] = (s, b)

    cal_table = pd.DataFrame(rows).sort_values(['SelectScore','RMSE','WAPE'], ascending=[True,True,True]).reset_index(drop=True)
    best_model = cal_table.iloc[0]['Model']
    return best_model, params[best_model], cal_table, models



def reduce_feature_cols(dataset_type, fit_df, feature_cols, max_external=60):
    """행이 적어서 외부변수는 fit 구간에서 target과 상관 높은 것만 사용."""
    internal, external, calendar = feature_groups(feature_cols)
    if dataset_type == 'internal_only':
        return feature_cols
    ext_candidates = [c for c in feature_cols if c.startswith('EXT_')]
    non_ext = [c for c in feature_cols if not c.startswith('EXT_')]
    if len(ext_candidates) <= max_external:
        return feature_cols
    temp = fit_df[ext_candidates + ['firm_quantity']].replace([np.inf, -np.inf], np.nan)
    scores = []
    for c in ext_candidates:
        sub = temp[[c, 'firm_quantity']].dropna()
        if len(sub) >= 10 and sub[c].nunique() > 1:
            corr = sub[c].corr(sub['firm_quantity'])
            if pd.notna(corr):
                scores.append((c, abs(corr)))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    selected_ext = [c for c, _ in scores[:max_external]]
    return non_ext + selected_ext

def run_one_dataset(dataset_type, fit, cal, train_eval, test, usable, future_feat, all_feature_cols):
    feature_cols_raw = get_feature_set(all_feature_cols, dataset_type)
    feature_cols = reduce_feature_cols(dataset_type, fit, feature_cols_raw, max_external=60)
    selected_model, (scale, offset), cal_table, _ = choose_best_by_cal(dataset_type, fit, cal, feature_cols)

    # holdout: train_eval 전체로 재학습 후, calibration에서 정한 selected_model/scale/offset 적용
    holdout_cand, _ = predict_candidates(dataset_type, train_eval, test, feature_cols)
    holdout_rows = []
    applied_preds = {}
    for col in [c for c in holdout_cand.columns if c != 'week']:
        # 성능표에는 각 후보 원값도 같이 보여줌. 최종 선택모델만 calibration scale/offset 적용.
        if col == selected_model:
            s, b = scale, offset
        else:
            s, b = 1.0, 0.0
        pred = pack_round(np.maximum(np.asarray(holdout_cand[col]) * s + b, 0))
        applied_preds[col] = pred
        row = eval_row(dataset_type, col, test['firm_quantity'], pred, train_eval['firm_quantity'], s, b)
        row['Selected'] = (col == selected_model)
        holdout_rows.append(row)

    holdout_table = pd.DataFrame(holdout_rows).sort_values(['Selected','RMSE','WAPE'], ascending=[False,True,True]).reset_index(drop=True)
    selected_pred = applied_preds[selected_model]

    holdout_detail = test[['week','firm_quantity','forecast_quantity']].copy()
    holdout_detail['dataset_type'] = dataset_type
    holdout_detail['selected_model'] = selected_model
    holdout_detail['pred'] = selected_pred
    holdout_detail['abs_error'] = np.abs(holdout_detail['firm_quantity'] - holdout_detail['pred'])
    holdout_detail['scale'] = scale
    holdout_detail['offset'] = offset

    # 미래 4주: usable 전체로 재학습 후 selected_model 적용
    future_cand, _ = predict_candidates(dataset_type, usable, future_feat, feature_cols)
    if selected_model in future_cand.columns:
        future_raw = np.asarray(future_cand[selected_model], dtype=float)
    else:
        # fallback
        future_raw = np.zeros(len(future_feat))
    future_pred = pack_round(np.maximum(future_raw * scale + offset, 0))
    future_forecast = pd.DataFrame({
        'week': future_feat['week'].values,
        'dataset_type': dataset_type,
        'selected_model': selected_model,
        'forecast_firm_quantity': future_pred,
        'raw_before_calibration': future_raw,
        'scale': scale,
        'offset': offset,
    })

    selected_holdout = holdout_table[holdout_table['Selected'] == True].iloc[0].to_dict()
    selected_summary = {
        'Dataset_Type': dataset_type,
        'Selected_Model': selected_model,
        'Feature_Count': len(feature_cols),
        'Scale': scale,
        'Offset': offset,
        'Holdout_RMSE': selected_holdout['RMSE'],
        'Holdout_WAPE': selected_holdout['WAPE'],
        'Holdout_MAE': selected_holdout['MAE'],
        'Holdout_Sum_Gap_Rate': selected_holdout['Sum Gap Rate'],
        'Actual Sum': selected_holdout['Actual Sum'],
        'Pred Sum': selected_holdout['Pred Sum'],
    }

    return {
        'dataset_type': dataset_type,
        'feature_cols': feature_cols,
        'selected_model': selected_model,
        'scale': scale,
        'offset': offset,
        'cal_table': cal_table,
        'holdout_table': holdout_table,
        'holdout_detail': holdout_detail,
        'future_forecast': future_forecast,
        'summary': selected_summary,
    }

# ------------------------------------------------------------
# 5. 전체 실행
# ------------------------------------------------------------
def run():
    ensure_files_exist()
    weekly, raw, firm_col, confirmed_weeks = load_demand_weekly(DEMAND_PATH)
    weekly_ext, ext_cols = load_external_weekly(EXTERNAL_PATH)
    base, future_weeks = make_base(weekly, weekly_ext)
    feat = engineer_features(base, ext_cols)

    confirmed = feat[feat['week'].isin(confirmed_weeks)].copy()
    required = ['firm_lag1','firm_lag2','firm_lag3','firm_lag4','firm_ma4','firm_ma8','forecast_quantity']
    usable = confirmed.dropna(subset=required).copy().reset_index(drop=True)
    if len(usable) < 20:
        raise ValueError(f'학습 가능한 FIRM 확정 주차가 너무 적습니다: {len(usable)}')

    test_n = min(HOLDOUT_CONFIRMED_WEEKS, max(4, len(usable)//5))
    train_eval = usable.iloc[:-test_n].copy()
    test = usable.iloc[-test_n:].copy()
    cal_n = min(INNER_CAL_CONFIRMED_WEEKS, max(3, len(train_eval)//5))
    fit = train_eval.iloc[:-cal_n].copy()
    cal = train_eval.iloc[-cal_n:].copy()

    exclude = {'week','firm_quantity'}
    all_feature_cols = [c for c in usable.columns if c not in exclude and pd.api.types.is_numeric_dtype(usable[c])]
    internal_cols, external_cols, calendar_cols = feature_groups(all_feature_cols)

    future_feat = feat[feat['week'].isin(future_weeks)].copy()

    dataset_types = ['internal_only', 'external_only', 'combined']
    results = []
    for ds in dataset_types:
        print(f'\n--- {ds} 모델 학습/평가 중 ---')
        results.append(run_one_dataset(ds, fit, cal, train_eval, test, usable, future_feat, all_feature_cols))

    summary = pd.DataFrame([r['summary'] for r in results]).sort_values(['Holdout_RMSE','Holdout_WAPE']).reset_index(drop=True)
    all_cal = pd.concat([r['cal_table'] for r in results], ignore_index=True)
    all_holdout = pd.concat([r['holdout_table'] for r in results], ignore_index=True)
    all_detail = pd.concat([r['holdout_detail'] for r in results], ignore_index=True)
    all_future = pd.concat([r['future_forecast'] for r in results], ignore_index=True)

    best_holdout_by_dataset = (
        all_holdout.sort_values(['Dataset_Type','RMSE','WAPE','MAE'])
        .groupby('Dataset_Type', as_index=False)
        .head(1)
        .reset_index(drop=True)
    )

    feature_count = pd.DataFrame([
        {'Dataset_Type':'internal_only','Feature_Count':len(get_feature_set(all_feature_cols,'internal_only'))},
        {'Dataset_Type':'external_only','Feature_Count':len(get_feature_set(all_feature_cols,'external_only'))},
        {'Dataset_Type':'combined','Feature_Count':len(get_feature_set(all_feature_cols,'combined'))},
        {'Dataset_Type':'internal_raw_count','Feature_Count':len(internal_cols)},
        {'Dataset_Type':'external_lag_count','Feature_Count':len(external_cols)},
        {'Dataset_Type':'calendar_count','Feature_Count':len(calendar_cols)},
    ])

    print('\n' + '='*115)
    print(f'[{TARGET_PN}] FIRM 확정 기준 v8 - 내부/외부/통합 3개 모델 비교')
    print('='*115)
    print(f'FIRM 원본 행 수: {len(raw[raw[firm_col].astype(str).str.upper().str.strip().eq(FIRM_VALUE)]):,}')
    print(f'FORECAST 원본 행 수: {len(raw[raw[firm_col].astype(str).str.upper().str.strip().eq(FORECAST_VALUE)]):,}')
    print(f'FIRM 확정 주 범위: {confirmed_weeks.min():%Y-%m-%d} ~ {confirmed_weeks.max():%Y-%m-%d}')
    print(f'예측 대상 4주: {future_weeks[0]:%Y-%m-%d} ~ {future_weeks[-1]:%Y-%m-%d}')
    print(f'외부변수 lag 범위: 1~{MAX_EXTERNAL_LAG}주')

    print('\n[Split 정보]')
    print(f'usable_n={len(usable)}, fit_n={len(fit)}, cal_n={len(cal)}, test_n={len(test)}')
    print(f'fit_range: {fit.week.min():%Y-%m-%d} ~ {fit.week.max():%Y-%m-%d}')
    print(f'cal_range: {cal.week.min():%Y-%m-%d} ~ {cal.week.max():%Y-%m-%d}')
    print(f'test_range: {test.week.min():%Y-%m-%d} ~ {test.week.max():%Y-%m-%d}')

    print('\n[Feature 개수]')
    print(feature_count.to_string(index=False))

    print('\n[운영 기준: calibration으로 선택된 모델 요약]')
    print(summary[['Dataset_Type','Selected_Model','Feature_Count','Holdout_RMSE','Holdout_WAPE','Holdout_MAE','Holdout_Sum_Gap_Rate','Actual Sum','Pred Sum','Scale','Offset']].to_string(index=False))

    print('\n[성능 비교 기준: holdout에서 가장 좋은 모델 1개씩]')
    print(best_holdout_by_dataset[['Dataset_Type','Model','RMSE','WAPE','MAE','Sum Gap Rate','Actual Sum','Pred Sum','Scale','Offset']].to_string(index=False))

    print('\n[Holdout 성능 전체 TOP 20]')
    show_cols = ['Dataset_Type','Selected','Model','RMSE','WAPE','MAE','Sum Gap Rate','Actual Sum','Pred Sum','Scale','Offset']
    print(all_holdout.sort_values(['RMSE','WAPE']).head(20)[show_cols].to_string(index=False))

    print('\n[미래 4주 예측 - 데이터셋별]')
    print(all_future.to_string(index=False))

    # 그래프: holdout actual vs 각 데이터셋 selected prediction
    plt.figure(figsize=(14,6))
    plt.plot(test['week'], test['firm_quantity'], marker='o', linewidth=3, label='Actual firm_quantity')
    for r in results:
        d = r['holdout_detail']
        label = f"{r['dataset_type']} / {r['selected_model']}"
        plt.plot(d['week'], d['pred'], marker='o', linestyle='--', label=label)
    plt.title(f'{TARGET_PN} Holdout: Internal vs External vs Combined')
    plt.xlabel('Week')
    plt.ylabel('firm_quantity')
    plt.grid(True, alpha=.3)
    plt.legend()
    plt.xticks(rotation=45)
    plt.tight_layout()
    graph_path = f'{OUTPUT_PREFIX}_holdout_compare_graph.png'
    plt.savefig(graph_path, dpi=200)
    plt.show()

    excel_path = f'{OUTPUT_PREFIX}_results.xlsx'
    split_info = pd.DataFrame({
        'item':['usable_n','fit_n','cal_n','test_n','fit_range','cal_range','test_range','external_lag_range','pack_unit'],
        'value':[len(usable),len(fit),len(cal),len(test),f'{fit.week.min():%Y-%m-%d}~{fit.week.max():%Y-%m-%d}',f'{cal.week.min():%Y-%m-%d}~{cal.week.max():%Y-%m-%d}',f'{test.week.min():%Y-%m-%d}~{test.week.max():%Y-%m-%d}',f'1~{MAX_EXTERNAL_LAG}',PACK_UNIT]
    })
    with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
        split_info.to_excel(writer, sheet_name='split_info', index=False)
        feature_count.to_excel(writer, sheet_name='feature_count', index=False)
        summary.to_excel(writer, sheet_name='cal_selected_summary', index=False)
        best_holdout_by_dataset.to_excel(writer, sheet_name='best_holdout_by_dataset', index=False)
        all_cal.to_excel(writer, sheet_name='calibration_all', index=False)
        all_holdout.to_excel(writer, sheet_name='holdout_all', index=False)
        all_detail.to_excel(writer, sheet_name='holdout_detail', index=False)
        all_future.to_excel(writer, sheet_name='forecast_4weeks', index=False)
        weekly.to_excel(writer, sheet_name='weekly_firm_forecast', index=False)
        # feature 목록 저장
        for ds in dataset_types:
            cols = get_feature_set(all_feature_cols, ds)
            pd.DataFrame({'feature': cols}).to_excel(writer, sheet_name=f'{ds}_features'[:31], index=False)

    print('\n[저장 완료]')
    print(excel_path)
    print(graph_path)

    if IN_COLAB:
        try:
            files.download(excel_path)
            files.download(graph_path)
        except Exception:
            pass

    return {
        'cal_selected_summary': summary,
        'best_holdout_by_dataset': best_holdout_by_dataset,
        'feature_count': feature_count,
        'calibration_all': all_cal,
        'holdout_all': all_holdout,
        'holdout_detail': all_detail,
        'forecast_4weeks': all_future,
        'excel_path': excel_path,
        'graph_path': graph_path,
    }


result_v8 = run()
